# Enterprise Retail Site Intelligence with AI Agents

## Geospatial AI for Retail Location Analysis

This notebook demonstrates how AI agents can power enterprise retail site selection and competitive analysis using real-world geospatial data.

### Business Use Case:
Retail enterprises need to understand the competitive landscape when evaluating new store locations, optimizing existing networks, or analyzing market penetration. This agent automates that analysis.

### What You'll Learn:
- How AI agents translate business questions into spatial database queries
- How to access 100M+ commercial POIs from Foursquare Open Places
- Why human-in-the-loop approval matters for enterprise AI systems
- How SedonaDB accelerates geospatial queries with Rust and Apache Arrow

### The Workflow:
1. **Ask**: Pose a business question in natural language
2. **Interpret**: Agent determines the spatial query strategy
3. **Approve**: Human reviews and approves before execution
4. **Execute**: Agent runs optimized queries via SedonaDB
5. **Analyze**: Agent summarizes findings with business context
6. **Export**: Results delivered as GeoJSON or CSV for further analysis

### Data Source:
[Foursquare Open Places](https://docs.foursquare.com/data-products/docs/access-fsq-os-places) - 100M+ commercial POIs licensed under Apache 2.0.

Let's get started!

## 1. Installation (Run Once)

If you haven't installed the required packages, uncomment and run:

In [1]:
# Uncomment to install (first time only):
!pip install -r requirements.txt

# You'll need a free Groq API key from: https://console.groq.com

## 1.5 Foursquare Place Categories Reference

Foursquare uses a hierarchical category taxonomy with up to 6 levels. Categories use human-readable labels like:
`Dining and Drinking > Restaurant > Italian Restaurant`

**Categories Relevant to Retail Site Analysis:**
- **Retail**: `Grocery Store`, `Supermarket`, `Pharmacy`, `Clothing Store`, `Shopping Mall`, `Department Store`, `Discount Store`
- **Food & Beverage**: `Coffee Shop`, `Restaurant`, `Fast Food Restaurant`, `Cafe`, `Bakery`
- **Financial Services**: `Bank`, `ATM`, `Credit Union`
- **Services**: `Gas Station`, `Laundromat`, `Post Office`
- **Health**: `Hospital`, `Pharmacy`, `Urgent Care Center`, `Doctor`
- **Recreation**: `Gym`, `Park`, `Movie Theater`

Run the cell below to search categories or browse by theme.

In [2]:
# Import category utilities from our src package
from src import load_categories, search_categories, browse_categories_by_prefix, get_popular_categories

# Categories are loaded from the SedonaDB 'categories' view (created by setup_database)
# They will be available after cell 3 runs. For now, show popular categories.

print("=" * 70)
print("FOURSQUARE CATEGORY EXAMPLES")
print("=" * 70)

print("\nPopular categories by theme:")
popular = get_popular_categories()
for theme, cats in popular.items():
    print(f"   {theme}: {', '.join(cats)}")

print("\n" + "=" * 70)
print("\nTIP: After running the setup cell, use these functions to explore:")
print("   - search_categories('coffee')")
print("   - search_categories('restaurant')")
print("   - browse_categories_by_prefix('Dining and Drinking')")
print("   - browse_categories_by_prefix('Retail')")

/Users/linkedin/Documents/agentic/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


FOURSQUARE CATEGORY EXAMPLES

Popular categories by theme:
   food: Restaurant, Coffee Shop, Cafe, Bar, Fast Food Restaurant, Bakery, Pizza Place
   retail: Grocery Store, Supermarket, Pharmacy, Clothing Store, Bookstore, Shopping Mall
   services: Bank, ATM, Post Office, Gas Station, Laundromat
   health: Hospital, Doctor, Dentist, Veterinarian, Urgent Care Center
   education: School, University, Library
   recreation: Park, Museum, Movie Theater, Gym, Stadium
   lodging: Hotel, Hostel, Bed and Breakfast


TIP: After running the setup cell, use these functions to explore:
   - search_categories('coffee')
   - search_categories('restaurant')
   - browse_categories_by_prefix('Dining and Drinking')
   - browse_categories_by_prefix('Retail')


## 2. Import Libraries

Import all the tools we need for our geospatial AI agent.

In [3]:
# Import from our src package
from src import setup_database, create_agent_tools, AgentLogger, get_bounding_box

# LangChain - Framework for building AI agents
from langchain_groq import ChatGroq
from langchain.agents import create_agent

# Standard Python libraries
import os
from datetime import datetime

print("All libraries imported successfully!")

All libraries imported successfully!


## 3. Configure Environment and Connect to SedonaDB

### What's happening here:
- **AWS Configuration**: Foursquare Open Places data is publicly available on S3 (no account needed)
- **SedonaDB Connection**: High-performance spatial database engine built on Rust and Apache Arrow
- **Places + Categories**: Loads both the places data (100M+ POIs) and categories taxonomy from S3
- **Groq LLM**: Free, fast API with Llama 3 models - get your key at [console.groq.com](https://console.groq.com)

### Why SedonaDB?
- **Fast**: Written in Rust, uses Apache Arrow for columnar processing
- **Efficient**: Pushdown predicates to S3, only reads required data
- **Spatial-native**: Built-in support for ST_Intersects, ST_Distance, etc.

### Foursquare Schema:
The places data uses flat columns (no nested structs):
- `name`, `latitude`, `longitude` - Basic identity
- `address`, `locality`, `region`, `country` - Location info
- `fsq_category_labels` - Array of category strings like `["Dining and Drinking > Coffee Shop"]`
- `date_closed` - NULL if the place is still open

In [4]:
# Set up SedonaDB and load Foursquare Open Places data
# This connects to SedonaDB, loads places + categories from S3, and registers them as views
sd, places_df, FSQ_BASE_PATH = setup_database()

# Load categories into pandas for local searching
categories_df = load_categories(sd)

# Show the schema so we know what fields are available
print("\nPlaces table schema:")
print(places_df.schema)

# ========== GROQ LLM CONFIGURATION ==========
# Groq provides free, fast inference with Llama 3 models
# Get your free API key from: https://console.groq.com

print("\nConfiguring Groq LLM...")
GROQ_API_KEY = input("Enter your Groq API key: ").strip()
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Create the LLM - using Llama 3.3 70B for best tool calling support
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("Groq configured with model: llama-3.3-70b-versatile")
print(f"\nEnvironment configured!")
print(f"Ready to query Foursquare Open Places: {FSQ_BASE_PATH}")

SedonaDB connected successfully!

Loading Foursquare Open Places data from S3...
   Places: s3://fsq-os-places-us-east-1/release/dt=2025-09-09/places/parquet/
Places data loaded and registered as 'places' view
   Categories: s3://fsq-os-places-us-east-1/release/dt=2025-09-09/categories/parquet/
Categories data loaded and registered as 'categories' view
Loaded 1245 categories from Foursquare taxonomy

Places table schema:
SedonaSchema with 27 fields:
  fsq_place_id: utf8<Utf8View>
  name: utf8<Utf8View>
  latitude: float64<Float64>
  longitude: float64<Float64>
  address: utf8<Utf8View>
  locality: utf8<Utf8View>
  region: utf8<Utf8View>
  postcode: utf8<Utf8View>
  admin_region: utf8<Utf8View>
  post_town: utf8<Utf8View>
  po_box: utf8<Utf8View>
  country: utf8<Utf8View>
  date_created: utf8<Utf8View>
  date_refreshed: utf8<Utf8View>
  date_closed: utf8<Utf8View>
  tel: utf8<Utf8View>
  website: utf8<Utf8View>
  email: utf8<Utf8View>
  facebook_id: int64<Int64>
  instagram: utf8<Utf8Vi

## 4. Helper Function: Get Bounding Box for Location

### Spatial Concept: Bounding Box
A **bounding box** is a rectangular area defined by four coordinates:
- Minimum longitude (west edge)
- Minimum latitude (south edge)
- Maximum longitude (east edge)
- Maximum latitude (north edge)

Think of it as drawing a rectangle on a map around your area of interest.

In [5]:
# The get_bounding_box function is imported from src.geocoding
# Test it!
test_bbox = get_bounding_box("Portland")
print(f"Test: Seattle bounding box:")
for key, value in test_bbox.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

Test: Seattle bounding box:
  min_lon: -122.7192
  min_lat: 45.4752
  max_lon: -122.6291
  max_lat: 45.5653
  center_lon: -122.6742
  center_lat: 45.5202
  location_name: Portland, Multnomah County, Oregon, United States


## 5. Define Agent Tools

### What are Tools?
Tools are functions that the AI agent can use to accomplish tasks. Each tool:
- Has a clear name and description
- Takes specific inputs
- Returns structured outputs
- Can interact with external systems (databases, APIs, files)

We define three tools (in `src/tools.py`):
1. **query_places_by_location** - Query Foursquare for POIs by location and category
2. **analyze_results** - Perform statistical analysis on query results
3. **export_results** - Export data to GeoJSON or CSV

### Tool 1: Query Places by Location

Queries Foursquare Open Places for POIs in a specific area using bounding box filtering on `latitude`/`longitude` and category matching on the `fsq_category_labels` array.

In [6]:
# Agent tools are defined in src/tools.py
# They include:
#   - query_places_by_location: Query Foursquare Open Places for POIs
#   - analyze_results: Analyze query results
#   - export_results: Export data to GeoJSON or CSV
#
# The tools are created using create_agent_tools() which configures them
# with the database connection.
#
# See src/tools.py for the full implementation details.

print("Tools are defined in src/tools.py")
print("They will be created when we set up the agent.")

Tools are defined in src/tools.py
They will be created when we set up the agent.


### Tool 2: Analyze Results

This tool performs statistical analysis on query results.

In [7]:
# See src/tools.py for analyze_results implementation
print("analyze_results tool: defined in src/tools.py")

analyze_results tool: defined in src/tools.py


### Tool 3: Export Results

This tool exports query results to GeoJSON or CSV format.

In [8]:
# See src/tools.py for export_results implementation
print("export_results tool: defined in src/tools.py")

export_results tool: defined in src/tools.py


## 6. Human-in-the-Loop Function

### Why Human-in-the-Loop?
AI agents should not operate completely autonomously, especially when:
- Accessing external data or APIs
- Making decisions that affect real systems
- Performing actions that have costs (API calls, database queries)

Human approval ensures:
- **Transparency**: You see what the agent wants to do
- **Control**: You can stop unwanted actions
- **Learning**: You understand the agent's decision-making process

In [9]:
# Human-in-the-loop approval is defined in src/approval.py
# It pauses execution to ask for human confirmation before agent actions.
#
# The tools automatically use human_approval() before running queries,
# so you'll see approval prompts when the agent executes.
#
# See src/approval.py for the implementation.

print("human_approval function: defined in src/approval.py")

human_approval function: defined in src/approval.py


## 7. Logging System

### Why Logging?
Logging the agent's thought process is crucial for:
- **Debugging**: Understanding why the agent made certain decisions
- **Learning**: Reviewing how AI agents reason about problems
- **Transparency**: Auditing agent actions for compliance or research
- **Improvement**: Identifying patterns to improve agent behavior

In [10]:
# AgentLogger is defined in src/logging.py
# It tracks agent actions, decisions, and reasoning throughout a session.
#
# Usage:
#   logger = AgentLogger()
#   logger.log_thought("User asked about coffee shops")
#   logger.log_action("query_places", approved=True, result="Found 50 places")
#   logger.save_to_file()
#
# See src/logging.py for the full implementation.

print("AgentLogger class: defined in src/logging.py")

AgentLogger class: defined in src/logging.py


## 8. Create the AI Agent

### What is an AI Agent?
An AI agent is a system that:
1. **Receives** a goal or question
2. **Reasons** about how to achieve it
3. **Uses tools** to gather information or take actions
4. **Iterates** until the goal is achieved
5. **Responds** with results

Unlike a simple chatbot, an agent can:
- Break complex tasks into steps
- Use multiple tools in sequence
- Adapt its approach based on results

In [11]:
# Create agent tools with the database connection
# This initializes the tools defined in src/tools.py with our SedonaDB connection
tools = create_agent_tools(sd, categories_df)

# Create the system prompt - this guides the agent's behavior
system_prompt = """
You are an enterprise retail site intelligence analyst powered by AI.

Your job is to help retail strategists, real estate teams, and business analysts
understand the competitive landscape and commercial density of specific markets
using Foursquare Open Places data.

When responding to queries:
1. Break down the business question to identify the location and category needed
2. Use query_places_by_location to retrieve POI data (this requires human approval)
3. Use analyze_results to provide market insights
4. Frame results in business terms: market density, competitive saturation, coverage gaps
5. Offer to export results for integration with internal GIS or BI tools

Foursquare categories use human-readable names like "Grocery Store", "Coffee Shop", "Bank".
Common retail categories: Grocery Store, Supermarket, Pharmacy, Department Store, 
Discount Store, Clothing Store, Shopping Mall, Coffee Shop, Fast Food Restaurant.

Always explain findings in terms that business stakeholders can act on.
"""

# Create the agent using the LangChain API
agent_executor = create_agent(llm, tools, system_prompt=system_prompt)

print("AI Agent created and ready!")
print("\nThe agent has access to 3 tools:")
print("   1. query_places_by_location - Query Foursquare for POIs by location + category")
print("   2. analyze_results - Analyze market density and distribution")
print("   3. export_results - Export data to GeoJSON or CSV for GIS/BI tools")

AI Agent created and ready!

The agent has access to 3 tools:
   1. query_places_by_location - Query Foursquare for POIs by location + category
   2. analyze_results - Analyze market density and distribution
   3. export_results - Export data to GeoJSON or CSV for GIS/BI tools


## 9. Example: Retail Competitive Analysis in Portland, OR

Let's see the agent in action with a real enterprise use case: analyzing grocery store density in the Portland, Oregon market.

### What to Watch For:
1. **Agent reasoning**: How it interprets a business question as a spatial query
2. **Tool selection**: Which tools it chooses and in what order
3. **Human approval**: The agent pauses before executing database queries
4. **Business framing**: How it presents findings in actionable terms
5. **Logging**: Complete audit trail of the agent's decision process

In [12]:
# Initialize logger for this session
logger = AgentLogger()

# Enterprise retail analysis query
user_query = "Find all grocery stores in Portland, Oregon and analyze the market density"

print("="*70)
print("STARTING AGENT SESSION")
print("="*70)
print(f"\nUser Query: {user_query}\n")
print("="*70)

# Log the initial query
logger.log_thought(f"User asked: {user_query}")

try:
    # Run the agent using LangGraph
    result = agent_executor.invoke({"messages": [("user", user_query)]})
    
    # Extract the final answer from the last message
    final_answer = result['messages'][-1].content
    
    print("\n" + "="*70)
    print("AGENT RESPONSE")
    print("="*70)
    print(f"\n{final_answer}\n")
    print("="*70)
    
    # Log the summary
    logger.log_summary(final_answer)
    
except Exception as e:
    error_msg = f"Error during agent execution: {str(e)}"
    print(f"\nError: {error_msg}")
    logger.log_action("agent_execution", False, error_msg)

# Save logs
log_file = logger.save_to_file()

print("\n" + "="*70)
print("SESSION COMPLETE")
print("="*70)
print(f"Session log saved to: {log_file}")
print("\nYou can review this log file to see the complete agent reasoning process.")

Logger initialized - Session ID: 20260309_114247
STARTING AGENT SESSION

User Query: Find all grocery stores in Portland, Oregon and analyze the market density


Agent thinking: User asked: Find all grocery stores in Portland, Oregon and analyze the market density

Searching for category: 'Grocery Store'
Found 1 matching categories in taxonomy
   - Grocery Store

Looking up coordinates for 'Portland, Oregon'...
Found: Portland, Multnomah County, Oregon, United States
Bounding box: [-122.7192, 45.4752, -122.6291, 45.5653]

🤖 AGENT REQUESTING APPROVAL

Action: Query Foursquare Open Places for 'Grocery Store' in Portland, Oregon

Details:
Area: Portland, Multnomah County, Oregon, United States
Category: Grocery Store
Bounding box: -122.7192, 45.4752, -122.6291, 45.5653


✅ APPROVED - Proceeding...


Executing query on Foursquare Open Places via SedonaDB...

SQL Query:

SELECT DISTINCT
        fsq_place_id,
        name,
        latitude,
        longitude,
        address,
        localit

## 10. Try Your Own Analysis

Modify the query below to explore different markets and retail categories.

### Finding Categories

**Use the category search from Section 1.5:**
```python
search_categories('grocery')
search_categories('pharmacy')
search_categories('department store')
browse_categories_by_prefix('Retail')
```

### Enterprise Analysis Ideas:
- **Competitive Landscape**: "Find all Coffee Shops in Portland, Oregon" (e.g. Starbucks site selection)
- **Market Penetration**: "Find Pharmacies in downtown Portland and analyze the distribution"
- **White Space Analysis**: "Find Grocery Stores in Seattle and show me the coverage"
- **Category Density**: "Find Fast Food Restaurants in Portland, Oregon"
- **Financial Services**: "Find Banks in Portland, Oregon and analyze the market"
- **Cross-Market Comparison**: Run the same category in two different cities and compare

In [ ]:
# Initialize a new logger
logger = AgentLogger()

# YOUR QUERY HERE - Change this to explore different markets!
your_query = "Find all Coffee Shops in Portland, Oregon and tell me about the competitive landscape"

print("="*70)
print("STARTING NEW AGENT SESSION")
print("="*70)
print(f"\nYour Query: {your_query}\n")
print("="*70)

logger.log_thought(f"User asked: {your_query}")

try:
    # Run the agent using LangGraph
    result = agent_executor.invoke({"messages": [("user", your_query)]})
    final_answer = result['messages'][-1].content
    
    print("\n" + "="*70)
    print("AGENT RESPONSE")
    print("="*70)
    print(f"\n{final_answer}\n")
    print("="*70)
    
    logger.log_summary(final_answer)
    
except Exception as e:
    error_msg = f"Error: {str(e)}"
    print(f"\nError: {error_msg}")
    logger.log_action("agent_execution", False, error_msg)

log_file = logger.save_to_file()
print(f"\nSession log: {log_file}")

## 11. Next Steps and Extensions

Congratulations! You've successfully:
- Built an AI agent for enterprise retail site intelligence
- Queried 100M+ POIs from Foursquare Open Places via SedonaDB
- Implemented human-in-the-loop approval for enterprise governance
- Logged the complete agent reasoning chain for audit compliance
- Exported results for integration with GIS and BI tools

### Enterprise Extensions:

1. **Multi-Market Analysis**
   - Compare retail density across metro areas
   - Identify underserved markets with coverage gap analysis
   - Track market changes using Foursquare delta files

2. **Advanced Site Selection**
   - Calculate drive-time catchment areas
   - Score locations by proximity to complementary businesses
   - Analyze cannibalization risk for new store placements

3. **Integration with Enterprise Tools**
   - Connect exported GeoJSON to ArcGIS or QGIS
   - Feed results into Tableau or Power BI dashboards
   - Build automated market reports with scheduled queries

4. **Enhanced Agent Capabilities**
   - Add memory for multi-turn market research sessions
   - Implement chain-of-thought reasoning for complex site evaluations
   - Create specialized agents for different analysis types (competitive, demographic, traffic)

### Resources:
- [Foursquare Open Places Documentation](https://docs.foursquare.com/data-products/docs/access-fsq-os-places)
- [Foursquare Open Places on HuggingFace](https://huggingface.co/datasets/foursquare/fsq-os-places)
- [SedonaDB Documentation](https://sedona.apache.org/sedonadb/)
- [LangChain Documentation](https://docs.langchain.com/)